# Triton Weighted Sum Tutorial

This notebook walks through implementing a weighted sum operation in Triton, with visualizations of shapes and intermediate outputs.

In [ ]:
import torch
import triton
import triton.language as tl
from einops import rearrange
import matplotlib.pyplot as plt
import numpy as np

## 1. The Operation We're Implementing

**Weighted Sum:** For each row of `x`, multiply element-wise by `weight` and sum.

```
x:      [N, D]  (e.g., [64, 128])
weight: [D]     (e.g., [128])
output: [N]     (e.g., [64])

output[i] = sum(x[i, :] * weight)
```

In [1]:
# Pure PyTorch reference implementation
def weighted_sum_pytorch(x, weight):
    # pure python implementation
    return (weight * x).sum(axis=-1)

# Test it
N, D = 64, 128
x = torch.randn(N, D, device='cuda')
weight = torch.randn(D, device='cuda')

output = weighted_sum_pytorch(x, weight)
print(f"x shape:      {x.shape}")
print(f"weight shape: {weight.shape}")
print(f"output shape: {output.shape}")

NameError: name 'torch' is not defined

## 2. Visualizing the Tiling Strategy

We split the computation into tiles:
- **ROWS_TILE_SIZE**: How many rows each program handles (parallelized)
- **D_TILE_SIZE**: How many columns per loop iteration (sequential)

In [ ]:
# Tiling parameters
ROWS_TILE_SIZE = 16
D_TILE_SIZE = 32

n_programs = triton.cdiv(N, ROWS_TILE_SIZE)
n_d_iterations = triton.cdiv(D, D_TILE_SIZE)

print(f"Matrix shape: [{N}, {D}]")
print(f"ROWS_TILE_SIZE: {ROWS_TILE_SIZE}")
print(f"D_TILE_SIZE: {D_TILE_SIZE}")
print(f"Number of programs (parallel): {n_programs}")
print(f"D iterations per program (sequential): {n_d_iterations}")

In [ ]:
# Visualize the tiling
fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.Set3(np.linspace(0, 1, n_programs))

for prog_id in range(n_programs):
    row_start = prog_id * ROWS_TILE_SIZE
    for d_idx in range(n_d_iterations):
        col_start = d_idx * D_TILE_SIZE
        rect = plt.Rectangle((col_start, row_start), D_TILE_SIZE, ROWS_TILE_SIZE,
                              facecolor=colors[prog_id], edgecolor='black', alpha=0.7)
        ax.add_patch(rect)
        ax.text(col_start + D_TILE_SIZE/2, row_start + ROWS_TILE_SIZE/2, 
                f'P{prog_id}\niter {d_idx}', ha='center', va='center', fontsize=8)

ax.set_xlim(0, D)
ax.set_ylim(0, N)
ax.set_xlabel('D dimension (columns) - sequential within program')
ax.set_ylabel('Rows - parallelized across programs')
ax.set_title('Triton Tiling: Each color = one program')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Understanding Strides

Strides tell us how to navigate memory.

In [ ]:
print(f"x shape: {x.shape}")
print(f"x.stride(0) = {x.stride(0)}  # Skip this many elements to go to next row")
print(f"x.stride(1) = {x.stride(1)}  # Skip this many elements to go to next column")
print()
print(f"weight shape: {weight.shape}")
print(f"weight.stride(0) = {weight.stride(0)}  # Elements are contiguous")
print()
print("To access x[row, col]: x_ptr + row * stride(0) + col * stride(1)")
print(f"Example: x[5, 10] = x_ptr + 5 * {x.stride(0)} + 10 * {x.stride(1)} = x_ptr + {5 * x.stride(0) + 10 * x.stride(1)}")

## 4. Simulating One Program's Work

Let's trace through what Program 0 does.

In [ ]:
# Simulate Program 0
program_id = 0
row_start = program_id * ROWS_TILE_SIZE
row_end = min(row_start + ROWS_TILE_SIZE, N)

print(f"Program {program_id} handles rows [{row_start}, {row_end})")
print()

# Initialize accumulator
accumulator = torch.zeros(ROWS_TILE_SIZE, device='cuda')

for d_iter in range(n_d_iterations):
    col_start = d_iter * D_TILE_SIZE
    col_end = min(col_start + D_TILE_SIZE, D)
    
    # Load tiles
    x_tile = x[row_start:row_end, col_start:col_end]  # [ROWS_TILE, D_TILE]
    w_tile = weight[col_start:col_end]                 # [D_TILE]
    
    # Compute partial sum
    partial = (x_tile * w_tile).sum(dim=1)  # [ROWS_TILE]
    
    # Pad if needed (simulating boundary check)
    if len(partial) < ROWS_TILE_SIZE:
        partial = torch.nn.functional.pad(partial, (0, ROWS_TILE_SIZE - len(partial)))
    
    accumulator += partial
    
    print(f"Iteration {d_iter}: cols [{col_start}, {col_end})")
    print(f"  x_tile shape: {x_tile.shape}")
    print(f"  w_tile shape: {w_tile.shape}")
    print(f"  partial sum shape: {partial.shape}")
    print()

print(f"Final accumulator shape: {accumulator.shape}")
print(f"These are the outputs for rows {row_start} to {row_end-1}")

## 5. Broadcasting: The Outer Product

In the backward pass, we compute `grad_x = grad_out[:, None] * weight[None, :]`

In [ ]:
# Simulate grad_out and weight for a tile
grad_out_tile = torch.randn(ROWS_TILE_SIZE, device='cuda')  # [16]
weight_tile = torch.randn(D_TILE_SIZE, device='cuda')        # [32]

print("Original shapes:")
print(f"  grad_out_tile: {grad_out_tile.shape}")
print(f"  weight_tile:   {weight_tile.shape}")
print()

# Add dimensions for broadcasting
grad_out_col = grad_out_tile[:, None]  # [16, 1] - column vector
weight_row = weight_tile[None, :]       # [1, 32] - row vector

print("After adding dimensions:")
print(f"  grad_out[:, None]: {grad_out_col.shape}")
print(f"  weight[None, :]:   {weight_row.shape}")
print()

# Outer product via broadcasting
grad_x_tile = grad_out_col * weight_row
print(f"Outer product result: {grad_x_tile.shape}")

## 6. The Triton Kernels

In [ ]:
@triton.jit
def weighted_sum_fwd(
    x_ptr, weight_ptr,
    output_ptr,
    x_stride_row, x_stride_dim,
    weight_stride_dim,
    output_stride_row,
    ROWS, D,
    ROWS_TILE_SIZE: tl.constexpr, D_TILE_SIZE: tl.constexpr,
):
    row_tile_idx = tl.program_id(0)

    x_block_ptr = tl.make_block_ptr(
        x_ptr,
        shape=(ROWS, D,),
        strides=(x_stride_row, x_stride_dim),
        offsets=(row_tile_idx * ROWS_TILE_SIZE, 0),
        block_shape=(ROWS_TILE_SIZE, D_TILE_SIZE),
        order=(1, 0),
    )

    weight_block_ptr = tl.make_block_ptr(
        weight_ptr,
        shape=(D,),
        strides=(weight_stride_dim,),
        offsets=(0,),
        block_shape=(D_TILE_SIZE,),
        order=(0,),
    )

    output_block_ptr = tl.make_block_ptr(
        output_ptr,
        shape=(ROWS,),
        strides=(output_stride_row,),
        offsets=(row_tile_idx * ROWS_TILE_SIZE,),
        block_shape=(ROWS_TILE_SIZE,),
        order=(0,),
    )

    output = tl.zeros((ROWS_TILE_SIZE,), dtype=tl.float32)

    for i in range(tl.cdiv(D, D_TILE_SIZE)):
        row = tl.load(x_block_ptr, boundary_check=(0, 1), padding_option="zero")
        weight = tl.load(weight_block_ptr, boundary_check=(0,), padding_option="zero")
        output += tl.sum(row * weight[None, :], axis=1)
        x_block_ptr = x_block_ptr.advance((0, D_TILE_SIZE))
        weight_block_ptr = weight_block_ptr.advance((D_TILE_SIZE,))

    tl.store(output_block_ptr, output, boundary_check=(0,))

In [ ]:
@triton.jit
def weighted_sum_backward(
    x_ptr, weight_ptr,
    grad_output_ptr,
    grad_x_ptr, partial_grad_weight_ptr,
    stride_xr, stride_xd,
    stride_wd,
    stride_gr,
    stride_gxr, stride_gxd,
    stride_gwb, stride_gwd,
    NUM_ROWS, D,
    ROWS_TILE_SIZE: tl.constexpr, D_TILE_SIZE: tl.constexpr,
):
    row_tile_idx = tl.program_id(0)
    n_row_tiles = tl.num_programs(0)

    grad_output_block_ptr = tl.make_block_ptr(
        grad_output_ptr,
        shape=(NUM_ROWS,), strides=(stride_gr,),
        offsets=(row_tile_idx * ROWS_TILE_SIZE,),
        block_shape=(ROWS_TILE_SIZE,),
        order=(0,),
    )

    x_block_ptr = tl.make_block_ptr(
        x_ptr,
        shape=(NUM_ROWS, D,), strides=(stride_xr, stride_xd),
        offsets=(row_tile_idx * ROWS_TILE_SIZE, 0),
        block_shape=(ROWS_TILE_SIZE, D_TILE_SIZE),
        order=(1, 0),
    )

    weight_block_ptr = tl.make_block_ptr(
        weight_ptr,
        shape=(D,), strides=(stride_wd,),
        offsets=(0,), block_shape=(D_TILE_SIZE,),
        order=(0,),
    )

    grad_x_block_ptr = tl.make_block_ptr(
        grad_x_ptr,
        shape=(NUM_ROWS, D,), strides=(stride_gxr, stride_gxd),
        offsets=(row_tile_idx * ROWS_TILE_SIZE, 0),
        block_shape=(ROWS_TILE_SIZE, D_TILE_SIZE),
        order=(1, 0),
    )

    partial_grad_weight_block_ptr = tl.make_block_ptr(
        partial_grad_weight_ptr,
        shape=(n_row_tiles, D,), strides=(stride_gwb, stride_gwd),
        offsets=(row_tile_idx, 0),
        block_shape=(1, D_TILE_SIZE),
        order=(1, 0),
    )

    for i in range(tl.cdiv(D, D_TILE_SIZE)):
        grad_output = tl.load(grad_output_block_ptr, boundary_check=(0,), padding_option="zero")
        weight = tl.load(weight_block_ptr, boundary_check=(0,), padding_option="zero")
        grad_x_row = grad_output[:, None] * weight[None, :]
        tl.store(grad_x_block_ptr, grad_x_row, boundary_check=(0, 1))

        row = tl.load(x_block_ptr, boundary_check=(0, 1), padding_option="zero")
        grad_weight_row = tl.sum(row * grad_output[:, None], axis=0, keep_dims=True)
        tl.store(partial_grad_weight_block_ptr, grad_weight_row, boundary_check=(1,))

        x_block_ptr = x_block_ptr.advance((0, D_TILE_SIZE))
        weight_block_ptr = weight_block_ptr.advance((D_TILE_SIZE,))
        partial_grad_weight_block_ptr = partial_grad_weight_block_ptr.advance((0, D_TILE_SIZE))
        grad_x_block_ptr = grad_x_block_ptr.advance((0, D_TILE_SIZE))

## 7. The Autograd Function

In [ ]:
class WeightedSumFunc(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight):
        D, output_dims = x.shape[-1], x.shape[:-1]
        input_shape = x.shape
        x = rearrange(x, "... d -> (...) d")

        ctx.save_for_backward(x, weight)

        assert len(weight.shape) == 1 and weight.shape[0] == D, "Dimension mismatch"
        assert x.is_cuda and weight.is_cuda, "Expected CUDA tensors"
        assert x.is_contiguous(), "Our pointer arithmetic will assume contiguous x"

        ctx.D_TILE_SIZE = triton.next_power_of_2(D) // 16
        ctx.ROWS_TILE_SIZE = 16
        ctx.input_shape = input_shape

        y = torch.empty(output_dims, device=x.device)
        n_rows = y.numel()
        
        weighted_sum_fwd[(triton.cdiv(n_rows, ctx.ROWS_TILE_SIZE),)](
            x, weight,
            y,
            x.stride(0), x.stride(1),
            weight.stride(0),
            y.stride(0),
            ROWS=n_rows, D=D,
            ROWS_TILE_SIZE=ctx.ROWS_TILE_SIZE, D_TILE_SIZE=ctx.D_TILE_SIZE,
        )

        return y.view(input_shape[:-1])

    @staticmethod
    def backward(ctx, grad_out):
        x, weight = ctx.saved_tensors
        ROWS_TILE_SIZE, D_TILE_SIZE = ctx.ROWS_TILE_SIZE, ctx.D_TILE_SIZE
        n_rows, D = x.shape

        partial_grad_weight = torch.empty((triton.cdiv(n_rows, ROWS_TILE_SIZE), D), device=x.device, dtype=x.dtype)
        grad_x = torch.empty_like(x)

        weighted_sum_backward[(triton.cdiv(n_rows, ROWS_TILE_SIZE),)](
            x, weight,
            grad_out,
            grad_x, partial_grad_weight,
            x.stride(0), x.stride(1),
            weight.stride(0),
            grad_out.stride(0),
            grad_x.stride(0), grad_x.stride(1),
            partial_grad_weight.stride(0), partial_grad_weight.stride(1),
            NUM_ROWS=n_rows, D=D,
            ROWS_TILE_SIZE=ROWS_TILE_SIZE, D_TILE_SIZE=D_TILE_SIZE,
        )

        grad_weight = partial_grad_weight.sum(axis=0)
        return grad_x, grad_weight


f_weightedsum = WeightedSumFunc.apply

## 8. Test It!

In [ ]:
# Create test inputs
x = torch.randn(64, 128, device='cuda', requires_grad=True)
w = torch.randn(128, device='cuda', requires_grad=True)

# Forward pass
y_triton = f_weightedsum(x, w)
y_pytorch = weighted_sum_pytorch(x, w)

print("Forward pass:")
print(f"  Triton output shape: {y_triton.shape}")
print(f"  PyTorch output shape: {y_pytorch.shape}")
print(f"  Max difference: {(y_triton - y_pytorch).abs().max().item():.2e}")
print(f"  Outputs match: {torch.allclose(y_triton, y_pytorch, atol=1e-5)}")

In [ ]:
# Backward pass
loss = y_triton.sum()
loss.backward()

print("Backward pass:")
print(f"  grad_x shape: {x.grad.shape}")
print(f"  grad_w shape: {w.grad.shape}")
print(f"  grad_fn: {y_triton.grad_fn}")

In [ ]:
# Verify gradients against PyTorch
x2 = x.detach().clone().requires_grad_(True)
w2 = w.detach().clone().requires_grad_(True)

y2 = weighted_sum_pytorch(x2, w2)
y2.sum().backward()

print("Gradient verification:")
print(f"  grad_x matches: {torch.allclose(x.grad, x2.grad, atol=1e-5)}")
print(f"  grad_w matches: {torch.allclose(w.grad, w2.grad, atol=1e-5)}")
print(f"  grad_x max diff: {(x.grad - x2.grad).abs().max().item():.2e}")
print(f"  grad_w max diff: {(w.grad - w2.grad).abs().max().item():.2e}")